# Day 4

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv('OPENAI_API_KEY'))

In [2]:
ret = llm.invoke('안녕하세요')
print(ret.content)

안녕하세요! 어떻게 도와드릴까요?


## 첫 번째 실습

### Memory / History

In [3]:
from langchain_core.chat_history import InMemoryChatMessageHistory

memory = InMemoryChatMessageHistory()

# - 1번 호출
memory.add_user_message('부산에서 가볼만한 관광지를 추천해주세요')

ret1 = llm.invoke(memory.messages)
memory.add_ai_message(ret1.content)

# - 2번 호출
memory.add_user_message('부산역에서 그곳에 가는 교통편을 알려주세요')

ret1 = llm.invoke(memory.messages)
memory.add_ai_message(ret1.content)

# - 3번 호출
memory.add_user_message('그 근처에서 먹을만한 음식점들을 추천해주세요')

ret1 = llm.invoke(memory.messages)
memory.add_ai_message(ret1.content)

for message in memory.messages:
    print(message)
print('-'*50)

# messages1.append(('ai',ret1.content))

content='부산에서 가볼만한 관광지를 추천해주세요' additional_kwargs={} response_metadata={}
content='부산은 아름다운 바다와 다양한 문화유산이 있는 도시입니다. 아래는 부산에서 가볼 만한 몇 가지 관광지를 추천드립니다.\n\n1. **해운대 해수욕장**: 부산을 대표하는 해수욕장으로, 해변에서 즐기는 여름의 즐거움과 주변의 다양한 레스토랑과 카페가 매력적입니다.\n\n2. **광안리 해수욕장**: 광안대교와 함께 아름다운 경치를 감상할 수 있는 곳으로, 야경이 특히 멋집니다. 다양한 해양 스포츠와 이벤트도 많이 열립니다.\n\n3. **부산 타워**: 부산의 전경을 한눈에 볼 수 있는 전망대로, 특히 해질녘이나 야경이 아름답습니다.\n\n4. **감천문화마을**: 파스텔 색상의 집들이 어우러져 있는 마을로, 독특한 예술적 디자인과 거리 풍경이 매력적입니다. 사진 찍기 좋은 곳이 많습니다.\n\n5. **자갈치 시장**: 부산의 대표적인 어시장으로, 신선한 해산물을 구매하거나 맛볼 수 있습니다. 현지인의 삶을 느껴볼 수 있는 장소입니다.\n\n6. **부산국제영화제(BIFF) 광장**: 영화제의 중심지로, 다양한 영화 관련 이벤트가 열리고, 주변에는 카페와 식당이 많아 즐길 거리가 많습니다.\n\n7. **태종대**: 아름다운 절경과 함께 자연을 느낄 수 있는 공원입니다. 산책로와 전망대가 있어 바다의 아름다움을 감상할 수 있습니다.\n\n8. **용두산 공원**: 부산 타워와 함께 있는 공원으로, 도시 중심부에 위치해 있어 편리합니다. 산책이나 피크닉을 즐기기 좋은 장소입니다.\n\n9. **부산시립미술관**: 현대 미술 작품을 감상할 수 있는 곳으로, 다양한 전시와 프로그램이 열립니다.\n\n이 외에도 부산은 많은 매력을 가진 도시이니, 여러 곳을 방문해 보시길 추천합니다!' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
memory = InMemoryChatMessageHistory()
memory.add_message(('system', '당신은 여행 전문가로 고객의 여행일정에 도움을 줍니다.'))

questions = [
    '부산에서 가볼만한 관광지를 추천해주세요',
    '부산역에서 그곳에 가는 교통편을 알려주세요',
    '그 근처에서 먹을만한 음식점들을 추천해주세요',
    '부산에서 가볼만한 돼지국밥집을 추천해주세요'
]
for question in questions:
    memory.add_user_message(question)
    ret = llm.invoke(memory.messages)
    memory.add_ai_message(ret.content)
    print(ret.content)
    print('-'*50)

for message in memory.messages:
    print(message)

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

cfg1 = {'configurable': {'session_id': 'user1'}}
cfg2 = {'configurable': {'session_id': 'user2'}}

store = {}

def get_memory(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 여행 전문가로 고객의 여행일정에 도움을 줍니다.'),
    MessagesPlaceholder('history'),
    ('user', '{input}')
])

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_memory,
    input_messages_key='input',
    history_messages_key='history'
)

ret = chain_with_history.invoke({'input': '부산 밤에 드라이브 할만한 해안가 추천해줘'}, config=cfg1)
print(ret.content)

In [ ]:
chain_with_history.invoke({'input': '안녕 클로드'}, config=cfg1)
chain_with_history.invoke({'input': '나는 홍길동이야'}, config=cfg2)

In [ ]:
for msg in store['user1'].messages:
    print(msg)

## 두 번째 실습
- 각각의 History를 갖는 3명의 대화를 만들어보세요

In [ ]:
store2 = {}

def get_memory2(session_id):
    if session_id not in store2:
        store2[session_id] = InMemoryChatMessageHistory()
    return store2[session_id]

prompt2 = ChatPromptTemplate.from_messages([
    ('system', '당신은 친절한 AI 비서입니다. 사용자가 알려준 정보를 기억하고 활용하세요.'),
    MessagesPlaceholder('history'),
    ('user', '{input}')
])

chain2 = prompt2 | llm

chat = RunnableWithMessageHistory(
    chain2,
    get_memory2,
    input_messages_key='input',
    history_messages_key='history'
)

cfg_user1 = {'configurable': {'session_id': 'user1'}}
cfg_user2 = {'configurable': {'session_id': 'user2'}}
cfg_user3 = {'configurable': {'session_id': 'user3'}}

# user1: 일본 여행
print('=== user1 ===')
print(chat.invoke({'input': '나는 일본 여행을 준비중이야.'}, config=cfg_user1).content)
print(chat.invoke({'input': '오사카와 교토를 방문할 예정이야.'}, config=cfg_user1).content)
print(chat.invoke({'input': '내가 어디를 여행하려고 했지?'}, config=cfg_user1).content)
print('-'*50)

# user2: 백화점
print('=== user2 ===')
print(chat.invoke({'input': '우리 백화점 이름은 행복백화점이야.'}, config=cfg_user2).content)
print(chat.invoke({'input': '이번 달 할인 행사는 여름 정기세일이야.'}, config=cfg_user2).content)
print(chat.invoke({'input': '우리 백화점 이름이 뭐였지?'}, config=cfg_user2).content)
print('-'*50)

# user3: 식당
print('=== user3 ===')
print(chat.invoke({'input': '우리 식당 이름은 맛있는집이야.'}, config=cfg_user3).content)
print(chat.invoke({'input': '대표 메뉴는 김치찌개야.'}, config=cfg_user3).content)
print(chat.invoke({'input': '우리 대표 메뉴가 뭐였지?'}, config=cfg_user3).content)

## 세 번째 실습 - window_size로 최근 대화만 기억하기

In [ ]:
store3 = {}

def get_window_memory(session_id):
    if session_id not in store3:
        store3[session_id] = InMemoryChatMessageHistory()
    return store3[session_id]

prompt3 = ChatPromptTemplate.from_messages([
    ('system', '당신은 친절한 AI 비서입니다. 사용자가 알려준 정보를 기억하고 활용하세요.'),
    MessagesPlaceholder('history'),
    ('user', '{input}')
])

chat3 = RunnableWithMessageHistory(
    prompt3 | llm,
    get_window_memory,
    input_messages_key='input',
    history_messages_key='history',
)

cfg = {'configurable': {'session_id': 'test'}}

questions = [
    '안녕하세요 챗봇님',
    '나는 홍길동입니다',
    '서울을 관광하고 있습니다. 서울 음식점 한 곳만 알려주세요',
    '그 곳의 제일 인기 있는 메뉴는 무엇인가요?',
    '서울역에서 그 곳까지 대중교통으로 가려면 어떻게 가야하나요?',
    '내 이름 기억하니?'
]

for q in questions:
    ret = chat3.invoke({'input': q}, config=cfg)
    print(f'[나] {q}')
    print(f'[AI] {ret.content}')
    print('-'*50)

In [ ]:
from openai import OpenAI
import os

client = OpenAI()

messages = [{"role": "user", "content": "서울 광장시장에서 가장 맛있는 길거리 음식은 무엇인가요?"}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0
)

print(response.choices[0].message.content)

usage = response.usage
print(f"Total Tokens: {usage.total_tokens}")
print(f"Prompt Tokens: {usage.prompt_tokens}")
print(f"Completion Tokens: {usage.completion_tokens}")
print(f"Total Cost (USD): ${usage.total_tokens * 0.00015 / 1000:.6f}")

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Person(BaseModel):
    name : str = Field(description='사람의 이름')
    age : int = Field(description='사람의 나이')

output = PydanticOutputParser(pydantic_object=Person)

In [ ]:
def myDeco(func):
    def wrapper():
        print('함수 시작')
        func()
        print('함수 종료')
    return wrapper

def hello():
    print('Hello, World!')

def hello2():
    print('안녕하세요')


hello()


print('함수 시작')

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

@tool
def add(x: int, y: int) -> int:
    ''' 두 숫자를 더하는 함수입니다.'''
    return x + y

@tool
def strlen(s: str) -> int:
    ''' 문자열의 길이를 반환하는 함수입니다.'''
    return len(s)

my_tools = [add, strlen]
tools_dict = { t.name : t for t in my_tools}

llm = ChatOpenAI(model='gpt-4o-mini')
llm_with_tools = llm.bind_tools(my_tools)

query = '무궁화 꽃이 피었습니다 문자열 길이를 알려줘'

#1차 llm 호출 - tool 판단
ret1 = llm_with_tools.invoke(query)
print(ret1.tool_calls)

if ret1.tool_calls :
    #tool을 호출 결과 result로 반환
    call = ret1.tool_calls[0]
    result = tools_dict[call['name']].invoke(call['args'])
    #2차 호출

else:
    print(ret1.content)

In [29]:
# tool 한 개 정의

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

@tool
def add(a: int, b: int) -> int:
    """두 숫자를 더합니다."""
    return a + b

@tool
def get_word_length(word: str) -> int:
    """단어나 문장의 정확한 글자 수(길이)를 계산할 때 사용합니다."""
    return len(word)

# LLM 생성
llm = ChatOpenAI(model='gpt-4o-mini')

tools=[add, get_word_length]
tools_dict = { t.name: t for t in tools }

# Tool 연결
llm_with_tool = llm.bind_tools(tools)

# 질문
query = "'만나서 반갑습니다'는 글자 수가 몇 개인지 알려주고, 그 결과에 5를 더해줘"

messages = [('user', query)]
for _ in range(10):
    res = llm_with_tool.invoke(messages)
    messages.append(res)
    if not res.tool_calls:
        print(res.content)
        break

    for call in res.tool_calls:
        result = tools_dict[call['name']].invoke(call['args'])
        print(f"\nTool '{call['name']}' 실행 결과: {result}")
        messages.append(ToolMessage(tool_call_id=call['id'], content=str(result)))
        print(messages)

print(res.content)


Tool 'get_word_length' 실행 결과: 9
[('user', "'만나서 반갑습니다'는 글자 수가 몇 개인지 알려주고, 그 결과에 5를 더해줘"), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 109, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0721cf3e68', 'id': 'chatcmpl-DsHhbjXrwKv8X6HtDn6HhjDLzZWgG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019edd5a-1d45-7f52-84b1-ac45b973f451-0', tool_calls=[{'name': 'get_word_length', 'args': {'word': '만나서 반갑습니다'}, 'id': 'call_Tjhw9VjhpHdMfKkguQftMOpA', 'type': 'tool_call'}, {'name': 'add', 'args': {'a': 5, 'b': 0}, 'id': 'call_u7CRhavZKDhF9LVvFOnbGB6s', 'type': 'tool_call'}], invalid_tool_calls=[